# 04 — Final Model Construction with Hyperparameter Optimization

> Clean senior-review version. This notebook contains only the final, logically connected pipeline. 
> Exploratory/probing/failed scripts are intentionally excluded from the main workflow.

## Objective
Train only final candidate models using the RFE-selected feature subset, then select the best model through cross-validated hyperparameter optimization. Exploratory files such as 4a/4b/4c are excluded from the main workflow.

In [ ]:
# !pip install -q pandas numpy scikit-learn xgboost joblib matplotlib

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

PROJECT_DIR = Path.cwd()
FS_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "feature_selection"
MODEL_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "final_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DATASET_FILE = FS_DIR / "recA_RFE_final_modeling_dataset.csv"
META_COLUMNS = ["Name", "SMILES", "EC50_nM", "pEC50", "bioactivity_class"]
RANDOM_STATE = 42

## 1. Load final RFE-selected dataset

In [ ]:
df = pd.read_csv(FINAL_DATASET_FILE)
feature_cols = [c for c in df.columns if c not in META_COLUMNS]

X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
y = df["bioactivity_class"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)
print("Final RFE features:", len(feature_cols))
print(y.value_counts())

## 2. Define models and hyperparameter search spaces

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

model_spaces = {
    "RandomForest": {
        "estimator": RandomForestClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        ),
        "params": {
            "n_estimators": [200, 300, 500, 800],
            "max_depth": [None, 5, 10, 20],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2", None],
        },
    },
    "SVM_RBF": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
        ]),
        "params": {
            "clf__C": [0.1, 1, 3, 10, 30, 100],
            "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.1],
            "clf__kernel": ["rbf"],
        },
    },
    "XGBoost": {
        "estimator": XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "n_estimators": [100, 200, 300, 500],
            "max_depth": [2, 3, 4, 5, 6],
            "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.10],
            "subsample": [0.70, 0.80, 0.90, 1.00],
            "colsample_bytree": [0.70, 0.80, 0.90, 1.00],
            "min_child_weight": [1, 3, 5],
            "reg_lambda": [1, 2, 5, 10],
        },
    },
}

## 3. Hyperparameter optimization

In [ ]:
search_results = []
best_models = {}

for name, spec in model_spaces.items():
    print(f"\nOptimizing {name}...")
    search = RandomizedSearchCV(
        estimator=spec["estimator"],
        param_distributions=spec["params"],
        n_iter=30,
        scoring="roc_auc",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True,
        verbose=1,
    )
    search.fit(X_train, y_train)

    best_models[name] = search.best_estimator_
    search_results.append({
        "model": name,
        "best_cv_roc_auc": search.best_score_,
        "best_params": search.best_params_,
    })

search_df = pd.DataFrame(search_results).sort_values("best_cv_roc_auc", ascending=False)
search_df.to_csv(MODEL_DIR / "hyperparameter_optimization_summary.csv", index=False)

search_df

## 4. Final evaluation on hold-out test set

In [ ]:
evaluation_rows = []

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)

    evaluation_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
    })

eval_df = pd.DataFrame(evaluation_rows).sort_values("roc_auc", ascending=False)
eval_df.to_csv(MODEL_DIR / "final_model_holdout_evaluation.csv", index=False)

BEST_MODEL_NAME = eval_df.iloc[0]["model"]
BEST_MODEL = best_models[BEST_MODEL_NAME]

print("Best final model:", BEST_MODEL_NAME)
eval_df

## 5. Save final trained model and feature list

In [ ]:
joblib.dump(BEST_MODEL, MODEL_DIR / "final_trained_model.joblib")

pd.DataFrame({"feature": feature_cols}).to_csv(MODEL_DIR / "final_model_features.csv", index=False)

final_metadata = {
    "best_model": BEST_MODEL_NAME,
    "selection_basis": "highest hold-out ROC-AUC after cross-validated hyperparameter optimization",
    "feature_source": "RFE-selected features from notebook 03",
    "n_features": len(feature_cols),
}
(MODEL_DIR / "final_model_metadata.json").write_text(json.dumps(final_metadata, indent=2), encoding="utf-8")

print(f"Saved final model: {MODEL_DIR / 'final_trained_model.joblib'}")
print(f"Saved feature list: {MODEL_DIR / 'final_model_features.csv'}")

## Final outputs
- `final_trained_model.joblib`
- `final_model_features.csv`
- `hyperparameter_optimization_summary.csv`
- `final_model_holdout_evaluation.csv`